# ClassicsRL: train pixel-based DQN on Colab

Use the same Python implementation as your Mac. Select **Runtime → Change runtime type → GPU**. Free GPU access and runtime duration are variable. This notebook saves verified training snapshots to Drive so a later session can continue from a fresh episode.

Download the source revision you intend to validate as a ZIP from GitHub and upload it below; no GitHub token is needed by this notebook. The ZIP must contain `pyproject.toml`, `src/`, and `configs/`. The Atari extra installs ALE, whose package includes game ROMs; review docs/game-content.md and applicable upstream terms before using it. Clear outputs before sharing this template.


In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import files

# If the in-cell uploader is unavailable, upload through Colab's Files sidebar
# and set SOURCE_ZIP to that file's /content/... path.
SOURCE_ZIP = None
if SOURCE_ZIP is None:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload one ClassicsRL source ZIP")
    archive_name = next(iter(uploaded))
else:
    archive_name = SOURCE_ZIP
source_root = Path("/content/classicsrl-source")
with zipfile.ZipFile(archive_name) as archive:
    archive.extractall(source_root)
projects = list(source_root.rglob("pyproject.toml"))
if len(projects) != 1:
    raise ValueError("Expected one project in the ZIP")
project = projects[0].parent
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{project}[atari,video]"], check=True
)
os.chdir(project)
# Newly installed editable paths are not processed by an already running kernel.
sys.path.insert(0, str(project / "src"))

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before training")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", torch.cuda.get_device_properties(0).total_memory / 2**30)

## Persistent storage

Authorize Drive access when prompted. Replay sampling and active training stay on the runtime disk. Completed checkpoints, logs, recordings, and verified snapshots are periodically copied into a dedicated Drive directory. A disconnect can lose work after the last completed persistent snapshot.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
durable_root = Path("/content/drive/MyDrive/ClassicsRL")

## Train or continue

Start with `TARGET_STEPS = 4096` to measure throughput with gradients active. A long recipe uses a 100,000-frame replay arena; CPU RAM, GPU use, and serialization costs should be checked before increasing capacity. These settings are a starting recipe, not demonstrated Pong performance.

For recovery in a new session, set `RESUME_FROM` to a completed snapshot directory in Drive and increase `TARGET_STEPS`. The notebook copies the saved run to local storage. A training snapshot includes replay and optimizer state; a small `.pt` playback checkpoint alone cannot resume training.


In [ ]:
import json
import shutil
from dataclasses import replace
from datetime import UTC, datetime

from classicsrl.config import Config, load_config
from classicsrl.training import train

TARGET_STEPS = 4096
RESUME_FROM = None  # e.g. "/content/drive/MyDrive/ClassicsRL/<run_id>/snapshots/step-..."
active_run = Path("/content/classicsrl-runs") / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")

if RESUME_FROM:
    persistent_snapshot = Path(RESUME_FROM)
    shutil.copytree(persistent_snapshot.parent.parent, active_run)
    resume = active_run / "snapshots" / persistent_snapshot.name
    saved = json.loads((resume / "state.json").read_text())
    config = replace(Config(**saved["config"]), device="cuda", total_steps=TARGET_STEPS)
else:
    resume = None
    # Use the same short recipe as the local CPU/MPS benchmark.
    recipe = "benchmark.toml" if TARGET_STEPS <= 4096 else "pong.toml"
    config = replace(
        load_config(project / "configs" / recipe), device="cuda", total_steps=TARGET_STEPS
    )

result = train(config, active_run, resume=resume, persist_dir=durable_root)
print("Durable run:", durable_root / result["run_id"])
print("Resume snapshot:", durable_root / result["run_id"] / "snapshots" / result["snapshot"].name)

## Inspect progress

Training loss and action values are diagnostics, not measures of playing strength. Use separate evaluation returns to assess the policy. The short throughput check may have no completed games and is not evidence of learning Pong.

Feature activity is measured on replay probes. A completely inactive convolution can block visual learning, while active features alone do not demonstrate useful play. Curves below keep resumed segments separate.


In [ ]:
import matplotlib.pyplot as plt

records = [json.loads(line) for line in (active_run / "metrics.jsonl").read_text().splitlines()]
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
metrics = ["train/td_loss", "train/features/conv3_active_fraction", "eval/mean_return"]
for axis, metric in zip(axes, metrics, strict=True):
    selected = [row for row in records if row["metric"] == metric]
    for segment in dict.fromkeys(row.get("segment_id", "legacy") for row in selected):
        points = [row for row in selected if row.get("segment_id", "legacy") == segment]
        axis.plot([row["agent_steps"] for row in points], [row["value"] for row in points])
    axis.set(xlabel="Agent steps", title=metric)
plt.tight_layout()

## Record an evaluation clip

This example caps a clip at 1,000 agent steps and logs that override explicitly. Video captures one original-color observation per decision (15 FPS with four-frame repetition), plus the final observation. A JSONL sidecar contains the matching action labels, values, and rewards.


In [ ]:
from IPython.display import Video, display

from classicsrl.artifacts import load_checkpoint, sync_run, write_json
from classicsrl.dqn import QNetwork
from classicsrl.evaluation import evaluate

saved = load_checkpoint(result["checkpoint"], "cuda")
network = QNetwork(saved["num_actions"]).to("cuda")
network.load_state_dict(saved["online"])
record_dir = active_run / "episodes" / f"preview-{datetime.now(UTC):%H%M%S%f}"
summary = evaluate(
    network, replace(config, max_episode_steps=1000), "cuda", [2_000_000], record_dir=record_dir
)
summary.update({"run_id": saved["run_id"], "checkpoint": result["checkpoint"].name})
write_json(record_dir / "evaluation.json", summary)
sync_run(active_run, durable_root / result["run_id"])
display(Video(str(record_dir / summary["episodes"][0]["video"]), embed=True))

Download the complete run directory from Drive for local inspection. Keep the directory structure intact. Exact emulator continuation and bitwise equality across CPU, MPS, and CUDA are not promised. See `docs/experiments.md` for measured results and limits. Free GPU availability varies.
